# TensorFlow Recommenders on recent movies (MovieLens 32M, cut to 1M ratings)

The same TFRS notebook as `notebooks/movielens/tfrs.ipynb`, on the dataset made by `01_prep.ipynb`: 4,000 movies
released 2010-2023, 10,628 users, ~1M ratings.

Two standard [TensorFlow Recommenders (TFRS)](https://www.tensorflow.org/recommenders) retrieval models
(`genrec.models.tfrs_models`), evaluated exactly like every other model:

| model | query tower ("who is asking") | order of ratings |
|---|---|---|
| **TFRS two-tower** | a learned vector per user (user-ID embedding) | ignored - like collaborative filtering, but learned |
| **TFRS sequential (GRU, last 20)** | a GRU reading the user's last 20 movies | used - comparable to SASRec-20 and the Qwen prompt |

Both are trained with TFRS's `Retrieval` task (in-batch softmax). Same split, the same 1,000 test users, full
ranking, already-rated movies skipped. The number of epochs is picked on the validation item.

**Runs in the separate TensorFlow environment** (kernel *Python (.venv-tf)*, see the README). CPU only, ~15 min.
TensorFlow training on CPU is not bit-for-bit repeatable: expect differences of ~0.01 HR@10 between runs.

In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import time

import pandas as pd
import tensorflow as tf
import tf_keras as keras

from genrec.data import Dataset, history, sample_test_users, target
from genrec.metrics import evaluate_scores, top_k
from genrec.models import tfrs_models as T
from genrec.paths import DATA_DIR, RESULTS_DIR

DATA = DATA_DIR / "ml-32m-recent"

MAX_EPOCHS = 30
PATIENCE = 3       # stop when validation NDCG@10 hasn't improved for this many epochs
CONTEXT = 20
SEED = 42
print(f"tensorflow {tf.__version__}, tf_keras {keras.__version__}")

tensorflow 2.21.0, tf_keras 2.21.0


## 1. Data

In [2]:
ds = Dataset.load(DATA)
seqs = ds.encoded()
users = ds.users
test_users = sample_test_users(users)
row_of = {u: r for r, u in enumerate(users)}
title = dict(zip(range(1, ds.index.n_cols), ds.movies.set_index("MovieID").loc[ds.index.movie_ids, "Title"]))


def split_lists(us, split):
    return [history(seqs[u], split) for u in us], [target(seqs[u], split) for u in us]


val_hist, val_tgt = split_lists(users, "val")


def fit_best(model, train_ds, score_fn, name):
    """Train epoch by epoch, keep the weights with the best validation NDCG@10 (all users)."""
    best, best_epoch, best_weights, t0 = -1.0, 0, None, time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        hist = model.fit(train_ds, epochs=1, verbose=0)
        val = evaluate_scores(score_fn(users, val_hist), val_hist, val_tgt)
        if val["NDCG@10"] > best:
            best, best_epoch, best_weights = val["NDCG@10"], epoch, model.get_weights()
        print(f"{name} epoch {epoch:2d}  loss {hist.history['loss'][0]:9.1f}  val HR@10 {val['HR@10']:.3f}  "
              f"NDCG@10 {val['NDCG@10']:.3f}  (best {best:.3f} @ {best_epoch})  {time.time() - t0:.0f}s")
        if epoch - best_epoch >= PATIENCE:
            break
    model.set_weights(best_weights)
    return best_epoch

## 2. Two-tower (user ID -> movie)

One embedding per user, so it only knows what the user rated during training: validation trains on `items[:-2]`,
the test model is retrained on `items[:-1]` for the number of epochs that was best on validation.

In [3]:
def two_tower():
    tf.random.set_seed(SEED)
    model = T.TwoTower(len(users), ds.index.n_cols)
    model.compile(optimizer=keras.optimizers.Adagrad(learning_rate=0.1))
    return model


twotower = two_tower()
best_epochs = fit_best(twotower, T.pairs_dataset(range(len(users)), val_hist),
                       lambda us, _: twotower.scores([row_of[u] for u in us]), "two-tower")
twotower = two_tower()
twotower.fit(T.pairs_dataset(range(len(users)), split_lists(users, "test")[0]), epochs=best_epochs, verbose=0)
print(f"two-tower retrained on items[:-1] for {best_epochs} epochs")

two-tower epoch  1  loss   29265.8  val HR@10 0.087  NDCG@10 0.044  (best 0.044 @ 1)  10s


two-tower epoch  2  loss   28696.3  val HR@10 0.093  NDCG@10 0.047  (best 0.047 @ 2)  19s


two-tower epoch  3  loss   28365.5  val HR@10 0.092  NDCG@10 0.046  (best 0.047 @ 2)  27s


two-tower epoch  4  loss   28124.2  val HR@10 0.090  NDCG@10 0.045  (best 0.047 @ 2)  36s


two-tower epoch  5  loss   27935.8  val HR@10 0.087  NDCG@10 0.043  (best 0.047 @ 2)  46s


two-tower retrained on items[:-1] for 2 epochs


## 3. Sequential: GRU over the last 20 movies -> next movie

Reads any history at prediction time, so no retraining for the test (like SASRec). Trained on every position of
`items[:-2]`.

In [4]:
tf.random.set_seed(SEED)
sequential = T.SequentialRetrieval(ds.index.n_cols, context=CONTEXT)
sequential.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))
train_ds, n_examples = T.sequence_dataset(val_hist, CONTEXT)
print(f"{n_examples} training examples")
fit_best(sequential, train_ds, lambda _, hist: sequential.scores(hist), "sequential")

967994 training examples


sequential epoch  1  loss    1574.1  val HR@10 0.087  NDCG@10 0.044  (best 0.044 @ 1)  45s


sequential epoch  2  loss    1524.9  val HR@10 0.100  NDCG@10 0.052  (best 0.052 @ 2)  89s


sequential epoch  3  loss    1488.0  val HR@10 0.111  NDCG@10 0.058  (best 0.058 @ 3)  132s


sequential epoch  4  loss    1457.1  val HR@10 0.117  NDCG@10 0.060  (best 0.060 @ 4)  179s


...

sequential epoch  6  loss    1421.6  val HR@10 0.123  NDCG@10 0.063  (best 0.063 @ 6)  269s


sequential epoch  7  loss    1411.1  val HR@10 0.122  NDCG@10 0.063  (best 0.063 @ 6)  315s


sequential epoch  8  loss    1403.3  val HR@10 0.122  NDCG@10 0.063  (best 0.063 @ 6)  364s


sequential epoch  9  loss    1397.8  val HR@10 0.121  NDCG@10 0.063  (best 0.063 @ 6)  409s


6

## 4. Test-set comparison

In [5]:
def test_scores(us):
    hist, _ = split_lists(us, "test")
    return {"TFRS two-tower": twotower.scores([row_of[u] for u in us]),
            f"TFRS sequential (GRU, last {CONTEXT})": sequential.scores(hist)}


test_hist, test_tgt = split_lists(test_users, "test")
scores = test_scores(test_users)
table = pd.DataFrame({name: evaluate_scores(s, test_hist, test_tgt) for name, s in scores.items()}).T
RESULTS_DIR.mkdir(exist_ok=True)
table.to_csv(RESULTS_DIR / "ml32m_recent_tfrs_test.csv")
print(f"test: last rating of {len(test_users)} users")
table.style.format("{:.3f}")

test: last rating of 1000 users


,hit@1,HR@10,NDCG@10
TFRS two-tower,0.018,0.097,0.051
"TFRS sequential (GRU, last 20)",0.021,0.107,0.056


In [6]:
# export the sequential model's top-10 + probabilities for the API's lookup tables
from genrec import lookup
from genrec.paths import ARTIFACTS_DIR

seq_name = f"TFRS sequential (GRU, last {CONTEXT})"
export = lookup.to_rows("tfrs-sequential", test_users, lookup.softmax_top_k(scores[seq_name], test_hist),
                        decode=lambda i: ds.index.movie_ids[i - 1])
EXPORT = ARTIFACTS_DIR / "ml32m-recent" / "tfrs_sequential.csv"
EXPORT.parent.mkdir(parents=True, exist_ok=True)
export.to_csv(EXPORT, index=False)
print(f"exported {len(export)} recommendations -> artifacts/ml32m-recent/{EXPORT.name}")

exported 10000 recommendations -> artifacts/ml32m-recent/tfrs_sequential.csv


### All users

In [7]:
all_hist, all_tgt = split_lists(users, "test")
pd.DataFrame({name: evaluate_scores(s, all_hist, all_tgt) for name, s in test_scores(users).items()}).T \
    .style.format("{:.3f}")

,hit@1,HR@10,NDCG@10
TFRS two-tower,0.014,0.092,0.046
"TFRS sequential (GRU, last 20)",0.022,0.112,0.059


## 5. Example recommendations

In [8]:
for r, u in enumerate(test_users[:3]):
    hist, tgt = test_hist[r], test_tgt[r]
    print(f"user {u}  last rated: {'; '.join(title[m] for m in hist[-5:])}")
    print(f"   real next: {title[tgt]}")
    for name, s in scores.items():
        top = top_k(s[r:r + 1], [hist])[0].tolist()
        verdict = f"HIT at rank {top.index(tgt) + 1}" if tgt in top else "miss"
        print(f"   {name[:15]:15s} ({verdict}): {'; '.join(title[m] for m in top[:5])} ...")
    print()

user 119156  last rated: The Martian (2015); Deadpool (2016); Star Wars: Episode VII - The Force Awakens (2015); Captain America: The Winter Soldier (2014); John Wick (2014)
   real next: Spotlight (2015)
   TFRS two-tower  (miss): Skyfall (2012); Hobbit: An Unexpected Journey, The (2012); Iron Man 3 (2013); Hobbit: The Desolation of Smaug, The (2013); Amazing Spider-Man, The (2012) ...
   TFRS sequential (miss): Ant-Man (2015); Captain America: Civil War (2016); Avengers: Age of Ultron (2015); Star Trek Into Darkness (2013); Kingsman: The Secret Service (2015) ...

user 130513  last rated: Safety Not Guaranteed (2012); Inception (2010); The Imitation Game (2014); Shutter Island (2010); Room (2015)
   real next: Misérables, Les (2012)
   TFRS two-tower  (miss): Giant Mechanical Man, The (2012); Your Sister's Sister (2011); Smashed (2012); Take This Waltz (2011); Art of Getting By, The (2011) ...
   TFRS sequential (miss): Big Short, The (2015); Spotlight (2015); Louis C.K.: Live at The